In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.models import resnet18, ResNet18_Weights

import torchmetrics

from tqdm.auto import tqdm
import numpy as np
import os

In [44]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")        # Apple Silicon (M1/M2/M3)
    elif torch.cuda.is_available():
        return torch.device("cuda")       # NVIDIA GPU
    else:
        return torch.device("cpu")

DEVICE = get_device()
print(f"utilisation de {DEVICE}")

utilisation de cuda


- moyenne et écart-type
- détermine la taille
- transforme en tenseurs
- normalisation

In [45]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

- batch taille de 64 images
- shuffle = true pour mélanger pendant l'entainement. c'est pas la peine pour le teste de validation

In [46]:
BATCH_SIZE  = 64
NUM_WORKERS = 0 

train_dataset = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
val_dataset   = datasets.CIFAR10('./data', train=False, download=True, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

CLASSES = train_dataset.classes   
print(f"Train : {len(train_dataset)} images | Val : {len(val_dataset)} images")
print(f"Classes : {CLASSES}")

Train : 50000 images | Val : 10000 images
Classes : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


- récupération de model préentrainré
- gèle les le réseau d'extraction et laisse la dèrenière douche pour fine-tuner la sortie
- 

In [50]:
class CIFAR10ResNet(nn.Module):

    def __init__(self, num_classes: int = 10, dropout_p: float = 0.4):
        super().__init__()

        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        for name, param in backbone.named_parameters():
            if not any(layer in name for layer in ["layer3", "layer4", "fc"]):
                param.requires_grad = False

        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity() 
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Flatten(),              # ← garantit (B, 512) quoi qu'il arrive
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, num_classes),
        )   
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)
model = CIFAR10ResNet(num_classes=10, dropout_p=0.4).to(DEVICE)


In [51]:
dummy = torch.randn(4, 3, 224, 224).to(DEVICE)
out   = model(dummy)
print(f"✅ Sortie modèle : {out.shape}")  # torch.Size([4, 10])

# Nombre de paramètres entraînables vs gelés
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres total : {total:,} | Entraînables : {trainable:,}")

✅ Sortie modèle : torch.Size([4, 10])
Paramètres total : 11,310,922 | Entraînables : 10,627,850


In [52]:
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [53]:
train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(DEVICE)
val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(DEVICE)
val_f1    = torchmetrics.F1Score(task="multiclass",  num_classes=10, average="macro").to(DEVICE)


In [54]:
writer = SummaryWriter(log_dir="runs/cifar10_resnet")

class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float("inf")
        self.stop      = False

    def step(self, val_loss: float):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

early_stopper = EarlyStopping(patience=5)

In [62]:
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0
    train_acc.reset()

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_acc.update(logits.argmax(dim=1), labels)
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss  = train_loss / len(train_loader)
    epoch_train_acc = train_acc.compute().item()

    model.eval()
    val_loss = 0.0
    val_acc.reset()
    val_f1.reset()

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, labels)
            val_loss += loss.item()
            val_acc.update(logits.argmax(dim=1), labels)
            val_f1.update(logits.argmax(dim=1), labels)

    avg_val_loss  = val_loss / len(val_loader)
    epoch_val_acc = val_acc.compute().item()
    epoch_val_f1  = val_f1.compute().item()

    scheduler.step()

    writer.add_scalars("Loss",     {"train": avg_train_loss, "val": avg_val_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": epoch_train_acc, "val": epoch_val_acc}, epoch)
    writer.add_scalar ("F1/val",   epoch_val_f1, epoch)
    writer.add_scalar ("LR",       scheduler.get_last_lr()[0], epoch)

    print(f"Epoch {epoch:02d} | "
          f"Train loss {avg_train_loss:.4f} acc {epoch_train_acc:.4f} | "
          f"Val loss {avg_val_loss:.4f} acc {epoch_val_acc:.4f} f1 {epoch_val_f1:.4f}")

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  💾 Meilleur modèle sauvegardé (acc={best_val_acc:.4f})")

    if early_stopper.step(avg_val_loss):
        print(f"⏹ Early stopping à l'epoch {epoch}")
        break

writer.close()
print(f"\n✅ Entraînement terminé. Meilleure val accuracy : {best_val_acc:.4f}")

Epoch 1/20 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [55]:
model.eval()
val_loss = 0.0
val_acc.reset()
val_f1.reset()

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss   = criterion(logits, labels)

        val_loss += loss.item()
        val_acc.update(logits.argmax(dim=1), labels)
        val_f1.update(logits.argmax(dim=1), labels)

avg_val_loss    = val_loss / len(val_loader)
epoch_val_acc   = val_acc.compute().item()
epoch_val_f1    = val_f1.compute().item()

scheduler.step()

Epoch 20/20 [Val]:   0%|          | 0/157 [00:00<?, ?it/s]

C:\Users\Malcolm\AppData\Local\Temp\ipykernel_13664\453741012.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


In [56]:
writer.add_scalars("Loss",     {"train": avg_train_loss, "val": avg_val_loss}, epoch)
writer.add_scalars("Accuracy", {"train": epoch_train_acc, "val": epoch_val_acc}, epoch)
writer.add_scalar ("F1/val",   epoch_val_f1, epoch)
writer.add_scalar ("LR",       scheduler.get_last_lr()[0], epoch)
if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_model.pth")

print(f"Epoch {epoch:02d} | "
        f"Train loss {avg_train_loss:.4f} acc {epoch_train_acc:.4f} | "
        f"Val loss {avg_val_loss:.4f} acc {epoch_val_acc:.4f} f1 {epoch_val_f1:.4f} | "
        f"LR {scheduler.get_last_lr()[0]:.2e}")

Epoch 20 | Train loss 0.0469 acc 0.9834 | Val loss 2.3556 acc 0.0998 f1 0.0271 | LR 9.94e-04


In [57]:
if early_stopper.step(avg_val_loss):
    print(f"⏹ Early stopping déclenché à l'epoch {epoch}")

writer.close()
print(f"\n✅ Entraînement terminé. Meilleure val accuracy : {best_val_acc:.4f}")


✅ Entraînement terminé. Meilleure val accuracy : 0.0998


In [59]:
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

model_cpu = model.to("cpu")

dummy_input = torch.randn(1, 3, 224, 224)   

torch.onnx.export(
    model_cpu,
    dummy_input,
    "cifar10_resnet18.onnx",
    input_names  = ["image"],
    output_names = ["logits"],
    dynamic_axes = {
        "image":  {0: "batch_size"},    
        "logits": {0: "batch_size"},
    },
    opset_version = 17,
    export_params = True,
)

print("✅ Modèle exporté : cifar10_resnet18.onnx")

C:\Users\Malcolm\AppData\Local\Temp\ipykernel_13664\388593338.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0611 11:15:06.415000 13664 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0611 11:15:06.888000 13664 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float'

[torch.onnx] Obtain model graph for `CIFAR10ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CIFAR10ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\Malcolm\Desktop\deap_learnin\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Malcolm\Desktop\deap_learnin\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
     

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


OSError: [Errno 22] Invalid argument: 'cifar10_resnet18.onnx.data'

In [ ]:
import onnx
onnx_model = onnx.load("cifar10_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("✅ Graphe ONNX valide")

# ── Inférence de test avec onnxruntime ───────────────────────────────────────
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("cifar10_resnet18.onnx",
                                providers=["CPUExecutionProvider"])

test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
outputs    = session.run(["logits"], {"image": test_input})
pred_class = np.argmax(outputs[0])
print(f"✅ Inférence ONNX OK — classe prédite : {CLASSES[pred_class]}")

✅ Graphe ONNX valide


NameError: name 'CLASSES' is not defined